# Notebook 02 — Treinamento do Modelo YOLOv8

## Objetivo
Treinar um modelo YOLOv8 para detectar soja (crop) e ervas daninhas (weed)
em imagens aéreas de lavoura.

## Dataset
- Nome: Weed Crop Aerial
- Fonte: Roboflow 100 / Intel
- Imagens: 1.176 (train: 823 | valid: 235 | test: 118)
- Classes: crop (0) | weed (1)
- Licença: CC BY 4.0

## Observações do Dataset (Notebook 01)
- Dataset desbalanceado: 5% crop vs 95% weed
- Objetos pequenos: maioria abaixo de 10% da imagem
- Algumas imagens com anotações incompletas

In [1]:
import os
from pathlib import Path

# Caminho base do projeto
BASE_DIR = Path("C:/projetos/soja-weed-detection")
DATASET_DIR = BASE_DIR / "notebooks" / "data" / "raw" / "weed-crop-aerial-1"
YAML_PATH = DATASET_DIR / "data.yaml"

# Verifica se tudo existe
print("=== Verificando caminhos ===")
print(f"Dataset: {DATASET_DIR}")
print(f"Existe: {DATASET_DIR.exists()}")
print(f"\nYAML: {YAML_PATH}")
print(f"Existe: {YAML_PATH.exists()}")

# Verifica as pastas de treino
for split in ["train", "valid", "test"]:
    pasta = DATASET_DIR / split / "images"
    qtd = len(os.listdir(pasta)) if pasta.exists() else 0
    print(f"\n{split}: {qtd} imagens — {pasta}")

=== Verificando caminhos ===
Dataset: C:\projetos\soja-weed-detection\notebooks\data\raw\weed-crop-aerial-1
Existe: True

YAML: C:\projetos\soja-weed-detection\notebooks\data\raw\weed-crop-aerial-1\data.yaml
Existe: True

train: 823 imagens — C:\projetos\soja-weed-detection\notebooks\data\raw\weed-crop-aerial-1\train\images

valid: 235 imagens — C:\projetos\soja-weed-detection\notebooks\data\raw\weed-crop-aerial-1\valid\images

test: 118 imagens — C:\projetos\soja-weed-detection\notebooks\data\raw\weed-crop-aerial-1\test\images


In [2]:
import yaml

# Lê o yaml original
with open(YAML_PATH, "r") as f:
    config = yaml.safe_load(f)

print("=== YAML original ===")
print(f"train: {config['train']}")
print(f"val:   {config['val']}")
print(f"test:  {config['test']}")

# Corrige para caminhos absolutos
config['train'] = str(DATASET_DIR / "train" / "images")
config['val']   = str(DATASET_DIR / "valid" / "images")
config['test']  = str(DATASET_DIR / "test"  / "images")

# Salva o yaml corrigido
yaml_corrigido = DATASET_DIR / "data_corrigido.yaml"
with open(yaml_corrigido, "w") as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print("\n=== YAML corrigido ===")
print(f"train: {config['train']}")
print(f"val:   {config['val']}")
print(f"test:  {config['test']}")
print(f"\nSalvo em: {yaml_corrigido}")

=== YAML original ===
train: ../train/images
val:   ../valid/images
test:  ../test/images

=== YAML corrigido ===
train: C:\projetos\soja-weed-detection\notebooks\data\raw\weed-crop-aerial-1\train\images
val:   C:\projetos\soja-weed-detection\notebooks\data\raw\weed-crop-aerial-1\valid\images
test:  C:\projetos\soja-weed-detection\notebooks\data\raw\weed-crop-aerial-1\test\images

Salvo em: C:\projetos\soja-weed-detection\notebooks\data\raw\weed-crop-aerial-1\data_corrigido.yaml


In [3]:
# Configurações do treinamento
config_treino = {
    "model":        "yolov8n.pt",    # nano — menor e mais rápido
    "data":         str(yaml_corrigido),
    "epochs":       50,              # voltas completas no dataset
    "imgsz":        640,             # tamanho da imagem
    "batch":        8,               # imagens por vez
    "patience":     10,              # para se não melhorar em 10 epochs
    "cls":          2.0,             # peso maior para erro de classe
    "project":      str(BASE_DIR / "models"),
    "name":         "soja_weed_v1",
    "exist_ok":     True,
}

print("=== Configuração do Treino ===")
for chave, valor in config_treino.items():
    print(f"{chave:12}: {valor}")

=== Configuração do Treino ===
model       : yolov8n.pt
data        : C:\projetos\soja-weed-detection\notebooks\data\raw\weed-crop-aerial-1\data_corrigido.yaml
epochs      : 50
imgsz       : 640
batch       : 8
patience    : 10
cls         : 2.0
project     : C:\projetos\soja-weed-detection\models
name        : soja_weed_v1
exist_ok    : True


In [4]:
from ultralytics import YOLO

print("=== Teste rápido — 1 epoch na CPU ===")
print("Isso vai demorar alguns minutos, é normal...\n")

# Carrega o modelo base
model = YOLO(config_treino["model"])

# Roda só 1 epoch para validar tudo
resultado = model.train(
    data     = config_treino["data"],
    epochs   = 1,
    imgsz    = config_treino["imgsz"],
    batch    = config_treino["batch"],
    project  = config_treino["project"],
    name     = config_treino["name"],
    exist_ok = config_treino["exist_ok"],
    device   = "cpu",
    verbose  = True,
)

print("\n=== Teste concluído! ===")
print("Se chegou aqui, tudo está funcionando.")
print("Agora vamos para o Google Colab com GPU.")

=== Teste rápido — 1 epoch na CPU ===
Isso vai demorar alguns minutos, é normal...

New https://pypi.org/project/ultralytics/8.4.52 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.47  Python-3.13.7 torch-2.11.0+cpu CPU (13th Gen Intel Core i7-1355U)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\projetos\soja-weed-detection\notebooks\data\raw\weed-crop-aerial-1\data_corrigido.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=Non